# Gold Sector Overview Mart

**Purpose**: Cross-sector market summary for executive dashboards.

**Target Table**: `workspace.gold.gold_sector_overview`

**Grain**: One row per date per sector with comprehensive market metrics

**Key Metrics**:
- Market size (active jobs, companies, locations)
- Growth indicators (new jobs, velocity)
- Competition metrics (jobs per company)
- Skill diversity
- Salary ranges

**Data Sources**:
- `workspace.warehouse.fact_job_postings`
- `workspace.warehouse.fact_salary`
- `workspace.warehouse.bridge_job_skill`

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.gold.gold_sector_overview
USING DELTA
COMMENT 'Cross-sector market overview for executive dashboards'
AS

WITH job_metrics AS (
  SELECT
    f.posting_date_sk AS sector_date_sk,
    f.sector_sk,
    
    -- MEASURES: Job counts
    COUNT(CASE WHEN f.active_flag = TRUE THEN 1 END) AS active_jobs,
    COUNT(CASE WHEN f.is_new_job = TRUE THEN 1 END) AS new_jobs,
    COUNT(CASE WHEN f.is_soft_delete = TRUE THEN 1 END) AS closed_jobs,
    
    -- MEASURES: Market size
    COUNT(DISTINCT CASE WHEN f.active_flag = TRUE THEN f.company_sk END) AS active_companies,
    COUNT(DISTINCT CASE WHEN f.active_flag = TRUE THEN f.role_sk END) AS active_roles,
    COUNT(DISTINCT CASE WHEN f.active_flag = TRUE THEN f.location_sk END) AS active_locations
    
  FROM workspace.warehouse.fact_job_postings f
  WHERE f.posting_date_sk >= CAST(DATE_FORMAT(DATE_SUB(CURRENT_DATE(), 365), 'yyyyMMdd') AS INT)
  GROUP BY f.posting_date_sk, f.sector_sk
),

skill_metrics AS (
  SELECT
    f.posting_date_sk,
    f.sector_sk,
    COUNT(DISTINCT bjs.skill_sk) AS unique_skills_demanded
  FROM workspace.warehouse.fact_job_postings f
  JOIN workspace.warehouse.bridge_job_skill bjs ON f.job_sk = bjs.job_sk
  WHERE f.posting_date_sk >= CAST(DATE_FORMAT(DATE_SUB(CURRENT_DATE(), 365), 'yyyyMMdd') AS INT)
    AND f.active_flag = TRUE
    AND bjs.is_current = TRUE
  GROUP BY f.posting_date_sk, f.sector_sk
),

salary_metrics AS (
  SELECT
    fs.observation_date_sk,
    fs.sector_sk,
    PERCENTILE(fs.salary_max, 0.50) AS median_salary_max,
    PERCENTILE(fs.salary_max, 0.25) AS p25_salary_max,
    PERCENTILE(fs.salary_max, 0.75) AS p75_salary_max,
    COUNT(*) AS salary_observations
  FROM workspace.warehouse.fact_salary fs
  WHERE fs.observation_date_sk >= CAST(DATE_FORMAT(DATE_SUB(CURRENT_DATE(), 365), 'yyyyMMdd') AS INT)
    AND fs.salary_period = 'ANNUAL'
    AND fs.salary_currency = 'USD'
    AND fs.salary_max > 0
  GROUP BY fs.observation_date_sk, fs.sector_sk
),

combined_metrics AS (
  SELECT
    jm.sector_date_sk,
    jm.sector_sk,
    jm.active_jobs,
    jm.new_jobs,
    jm.closed_jobs,
    jm.active_companies,
    jm.active_roles,
    jm.active_locations,
    COALESCE(sm.unique_skills_demanded, 0) AS unique_skills_demanded,
    COALESCE(sal.median_salary_max, 0) AS median_salary_max,
    COALESCE(sal.p25_salary_max, 0) AS p25_salary_max,
    COALESCE(sal.p75_salary_max, 0) AS p75_salary_max,
    COALESCE(sal.salary_observations, 0) AS salary_observations
  FROM job_metrics jm
  LEFT JOIN skill_metrics sm 
    ON jm.sector_date_sk = sm.posting_date_sk 
    AND jm.sector_sk = sm.sector_sk
  LEFT JOIN salary_metrics sal 
    ON jm.sector_date_sk = sal.observation_date_sk 
    AND jm.sector_sk = sal.sector_sk
),

temporal_enriched AS (
  SELECT
    cm.*,
    
    -- TEMPORAL METRICS: Prior period comparisons
    LAG(cm.active_jobs, 7) OVER (
      PARTITION BY cm.sector_sk
      ORDER BY cm.sector_date_sk
    ) AS prev_week_active_jobs,
    
    LAG(cm.active_jobs, 30) OVER (
      PARTITION BY cm.sector_sk
      ORDER BY cm.sector_date_sk
    ) AS prev_month_active_jobs,
    
    LAG(cm.median_salary_max, 30) OVER (
      PARTITION BY cm.sector_sk
      ORDER BY cm.sector_date_sk
    ) AS prev_month_median_salary,
    
    -- Month-to-date cumulative
    SUM(cm.new_jobs) OVER (
      PARTITION BY cm.sector_sk,
        YEAR(TO_DATE(CAST(cm.sector_date_sk AS STRING), 'yyyyMMdd')),
        MONTH(TO_DATE(CAST(cm.sector_date_sk AS STRING), 'yyyyMMdd'))
      ORDER BY cm.sector_date_sk
      ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS mtd_new_jobs,
    
    -- 30-day rolling averages
    AVG(cm.active_jobs) OVER (
      PARTITION BY cm.sector_sk
      ORDER BY cm.sector_date_sk
      ROWS BETWEEN 29 PRECEDING AND CURRENT ROW
    ) AS rolling_30day_avg_active_jobs
    
  FROM combined_metrics cm
)

SELECT
  -- DIMENSIONS
  te.sector_date_sk,
  te.sector_sk,
  
  -- MEASURES: Job metrics
  te.active_jobs,
  te.new_jobs,
  te.closed_jobs,
  te.active_companies,
  te.active_roles,
  te.active_locations,
  te.unique_skills_demanded,
  
  -- DERIVED: Market concentration
  CASE 
    WHEN te.active_companies > 0 
    THEN CAST(te.active_jobs AS DECIMAL(10,2)) / te.active_companies
    ELSE NULL 
  END AS jobs_per_company,
  
  CASE 
    WHEN te.active_roles > 0 
    THEN CAST(te.unique_skills_demanded AS DECIMAL(10,2)) / te.active_roles
    ELSE NULL 
  END AS skills_per_role,
  
  -- DERIVED: Market velocity
  CASE 
    WHEN te.active_jobs > 0 
    THEN CAST(te.new_jobs AS DECIMAL(10,4)) / te.active_jobs
    ELSE NULL 
  END AS hiring_velocity,
  
  -- MEASURES: Salary metrics
  CAST(te.median_salary_max AS DECIMAL(18,2)) AS median_salary_max,
  CAST(te.p25_salary_max AS DECIMAL(18,2)) AS p25_salary_max,
  CAST(te.p75_salary_max AS DECIMAL(18,2)) AS p75_salary_max,
  te.salary_observations,
  
  -- TEMPORAL METRICS
  te.mtd_new_jobs,
  CAST(te.rolling_30day_avg_active_jobs AS DECIMAL(10,2)) AS rolling_30day_avg_active_jobs,
  
  -- DERIVED: Period-over-period changes
  CASE 
    WHEN te.prev_week_active_jobs > 0 
    THEN CAST((te.active_jobs - te.prev_week_active_jobs) AS DECIMAL(10,4)) / te.prev_week_active_jobs
    ELSE NULL 
  END AS pct_change_jobs_vs_prev_week,
  
  CASE 
    WHEN te.prev_month_active_jobs > 0 
    THEN CAST((te.active_jobs - te.prev_month_active_jobs) AS DECIMAL(10,4)) / te.prev_month_active_jobs
    ELSE NULL 
  END AS pct_change_jobs_vs_prev_month,
  
  CASE 
    WHEN te.prev_month_median_salary > 0 
    THEN CAST((te.median_salary_max - te.prev_month_median_salary) AS DECIMAL(10,4)) / te.prev_month_median_salary
    ELSE NULL 
  END AS pct_change_salary_vs_prev_month,
  
  -- METADATA
  CURRENT_TIMESTAMP() AS created_at,
  UUID() AS batch_id
  
FROM temporal_enriched te
ORDER BY te.sector_date_sk DESC, te.active_jobs DESC;

In [0]:
%sql
-- Validation: Sector comparison
SELECT
  s.sector_name,
  s.sector_family,
  COUNT(DISTINCT gso.sector_date_sk) AS days_with_data,
  ROUND(AVG(gso.active_jobs), 0) AS avg_active_jobs,
  ROUND(AVG(gso.active_companies), 0) AS avg_active_companies,
  ROUND(AVG(gso.jobs_per_company), 2) AS avg_jobs_per_company,
  ROUND(AVG(gso.hiring_velocity), 4) AS avg_hiring_velocity,
  ROUND(AVG(gso.median_salary_max), 0) AS avg_median_salary,
  ROUND(AVG(gso.unique_skills_demanded), 0) AS avg_unique_skills
FROM workspace.gold.gold_sector_overview gso
JOIN workspace.warehouse.dim_sector s ON gso.sector_sk = s.sector_sk
GROUP BY s.sector_name, s.sector_family
ORDER BY avg_active_jobs DESC;